In [1]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q http://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install -q findspark

In [3]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"

In [4]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better
spark

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.types import ArrayType, StructField, StructType, StringType, IntegerType
from datetime import date,datetime
from pyspark.sql import Row

In [7]:
!unzip /content/titanic.zip


Archive:  /content/titanic.zip
  inflating: gender_submission.csv   
  inflating: test.csv                
  inflating: train.csv               


In [8]:
!ls

gender_submission.csv  spark-3.5.1-bin-hadoop3	    test.csv	 train.csv
sample_data	       spark-3.5.1-bin-hadoop3.tgz  titanic.zip


In [9]:
df = spark.read.csv("train.csv",header=True, inferSchema=True)

#inferSchema is uisng for description of col

In [ ]:
df.show()

In [12]:

df.printSchema() #type of variables


###########
#Var types on Pyspark


#StringType() = str

#IntegerType() = int

#LongType() = int

#DoubleType() = float

#FloatType() = float

#BooleanType() = bool

#DateType() = datetime.date

#TimestampType() = datetime.datetime

#ArrayType() = list

#MapType() = dict

#StructType() = dict ou object

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)



In [10]:
df.show(3, vertical=True) #Mostra cada linha no formato vertical (transposto)

df.select("Survived").show() #mostra apenas uma col


-RECORD 0---------------------------
 PassengerId | 1                    
 Survived    | 0                    
 Pclass      | 3                    
 Name        | Braund, Mr. Owen ... 
 Sex         | male                 
 Age         | 22.0                 
 SibSp       | 1                    
 Parch       | 0                    
 Ticket      | A/5 21171            
 Fare        | 7.25                 
 Cabin       | NULL                 
 Embarked    | S                    
-RECORD 1---------------------------
 PassengerId | 2                    
 Survived    | 1                    
 Pclass      | 1                    
 Name        | Cumings, Mrs. Joh... 
 Sex         | female               
 Age         | 38.0                 
 SibSp       | 1                    
 Parch       | 0                    
 Ticket      | PC 17599             
 Fare        | 71.2833              
 Cabin       | C85                  
 Embarked    | C                    
-RECORD 2---------------------------
 

In [14]:
df.select('*') # select all the columns


PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,0,3,"Braund, Mr. Owen ...",male,22.0,1,0,A/5 21171,7.25,NULL,S
2,1,1,"Cumings, Mrs. Joh...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. ...",female,26.0,0,0,STON/O2. 3101282,7.925,NULL,S
4,1,1,"Futrelle, Mrs. Ja...",female,35.0,1,0,113803,53.1,C123,S
5,0,3,"Allen, Mr. Willia...",male,35.0,0,0,373450,8.05,NULL,S
6,0,3,"Moran, Mr. James",male,NULL,0,0,330877,8.4583,NULL,Q
7,0,1,"McCarthy, Mr. Tim...",male,54.0,0,0,17463,51.8625,E46,S
8,0,3,"Palsson, Master. ...",male,2.0,3,1,349909,21.075,NULL,S
9,1,3,"Johnson, Mrs. Osc...",female,27.0,0,2,347742,11.1333,NULL,S
10,1,2,"Nasser, Mrs. Nich...",female,14.0,1,0,237736,30.0708,NULL,C


In [19]:
df.where((df.Age>25) & (df.Survived==1))

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
2,1,1,"Cumings, Mrs. Joh...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. ...",female,26.0,0,0,STON/O2. 3101282,7.925,NULL,S
4,1,1,"Futrelle, Mrs. Ja...",female,35.0,1,0,113803,53.1,C123,S
9,1,3,"Johnson, Mrs. Osc...",female,27.0,0,2,347742,11.1333,NULL,S
12,1,1,"Bonnell, Miss. El...",female,58.0,0,0,113783,26.55,C103,S
16,1,2,"Hewlett, Mrs. (Ma...",female,55.0,0,0,248706,16.0,NULL,S
22,1,2,"Beesley, Mr. Lawr...",male,34.0,0,0,248698,13.0,D56,S
24,1,1,"Sloper, Mr. Willi...",male,28.0,0,0,113788,35.5,A6,S
26,1,3,"Asplund, Mrs. Car...",female,38.0,1,5,347077,31.3875,NULL,S
53,1,1,"Harper, Mrs. Henr...",female,49.0,1,0,PC 17572,76.7292,D33,C


In [20]:
df.agg({'Age': 'max'})

max(Age)
80.0


In [24]:
df.groupBy('Pclass').agg({'Age': 'avg'}).orderBy('Pclass',ascending=False)

Pclass,avg(Age)
3,25.14061971830986
2,29.87763005780347
1,38.233440860215055


In [27]:
df.filter(df.Age>25).agg({'Fare':'avg'})

avg(Fare)
37.61960169491524


In [29]:
df.groupBy('Pclass').agg({'Fare':'avg'}).orderBy('Pclass',ascending=False)

Pclass,avg(Fare)
3,13.675550101832997
2,20.66218315217391
1,84.15468749999992


In [40]:
from pyspark.sql import SparkSession
from pyspark.sql.types import ArrayType, StructField, StructType, StringType, IntegerType
from datetime import date,datetime
from pyspark.sql import Row
from pyspark.sql.functions import udf

In [41]:
def round_float_down(x):
  return(int(x))

In [42]:
round_float_down_udf = udf(round_float_down, IntegerType())

In [49]:
df.select('PassengerId',round_float_down_udf('Fare')).alias('Fare Rounded Down').limit(7) #.show() #passa a col pra int e add uma nova col com o nome pro df

PassengerId,round_float_down(Fare)
1,7
2,71
3,7
4,53
5,8
6,8
7,51


In [50]:
df.createOrReplaceTempView('titanic')

In [53]:
spark.sql('select * from titanic where Age>25')

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
2,1,1,"Cumings, Mrs. Joh...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. ...",female,26.0,0,0,STON/O2. 3101282,7.925,NULL,S
4,1,1,"Futrelle, Mrs. Ja...",female,35.0,1,0,113803,53.1,C123,S
5,0,3,"Allen, Mr. Willia...",male,35.0,0,0,373450,8.05,NULL,S
7,0,1,"McCarthy, Mr. Tim...",male,54.0,0,0,17463,51.8625,E46,S
9,1,3,"Johnson, Mrs. Osc...",female,27.0,0,2,347742,11.1333,NULL,S
12,1,1,"Bonnell, Miss. El...",female,58.0,0,0,113783,26.55,C103,S
14,0,3,"Andersson, Mr. An...",male,39.0,1,5,347082,31.275,NULL,S
16,1,2,"Hewlett, Mrs. (Ma...",female,55.0,0,0,248706,16.0,NULL,S
19,0,3,"Vander Planke, Mr...",female,31.0,1,0,345763,18.0,NULL,S
